# 06 — Supervised ranker (walk-forward, walks 1-16)

LightGBM `LGBMRanker(lambdarank)` over PCA text + structured + macro + aux text
features.

**Structure:**
- Cells A-F — deep dive on **walk 1** (train 2002-2007, val 2008, test 2009):
  load PCA + assemble features, Optuna HP search (15 trials, val NDCG@30),
  final fit with early stopping, OOS evaluation, feature importance plot, persist.
- Cells G-H — **walks 2-16** loop using walk-1's frozen HPs (per spec §7.2),
  then cross-walk diagnostics aggregating all 16 walks' metrics.

Outputs land in `artifacts/ranker/walk-{N:03d}/` per walk and
`artifacts/ranker/all_walks_summary.json` consolidated.

**Spec:** `docs/superpowers/specs/2026-05-16-supervised-ranker-design.md`.

Per the `machine-learning` skill: Optuna study persisted, early stopping on val
NDCG@30, no tuning on test, model + HPs + diagnostics versioned together.

## A. Setup

In [ ]:
from __future__ import annotations
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import optuna
from lightgbm import early_stopping
import matplotlib.pyplot as plt

from src.utils.io import processed_dir, repo_root
from src.utils.ranker import (
    DEFAULT_RANKER_PARAMS,
    assemble_walk_features,
    build_ranker,
    compute_excess_return_buckets,
    drop_zero_info_columns,
    evaluate_ranker,
    load_walk_pca,
    project_text_to_pca,
)

WALK_ID = 1
TRAIN_START, TRAIN_END = '2002-01-01', '2007-12-31'
VAL_START,   VAL_END   = '2008-01-01', '2008-12-31'
TEST_START,  TEST_END  = '2009-01-01', '2009-12-31'

PANEL_DIR = processed_dir() / 'training_panel'
EMBED_DIR = processed_dir() / 'finbert_stockday_embed'
OUT_DIR   = repo_root() / 'artifacts' / 'ranker' / f'walk-{WALK_ID:03d}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_OPTUNA_TRIALS = 15
N_BUCKETS = 32
TOP_K = 30
RANDOM_STATE = 42

print(f'walk {WALK_ID}')
print(f'  train: {TRAIN_START} .. {TRAIN_END}')
print(f'  val:   {VAL_START} .. {VAL_END}')
print(f'  test:  {TEST_START} .. {TEST_END}')
print(f'out_dir: {OUT_DIR.relative_to(repo_root())}')

## B. Load PCA + assemble train/val/test feature matrices

Reads the per-year shards in the three windows, projects text vecs through the
walk-1 PCA (n_pca=79), and assembles (X, y_excess, group_sizes, meta) triples
via `assemble_walk_features`. Memory ceiling: ~470k train rows × ~196 cols ≈ 700 MB.

In [ ]:
pca, n_pca = load_walk_pca(WALK_ID)
print(f'PCA loaded: n_components = {n_pca}')


def _load_panel_years(start: str, end: str) -> pd.DataFrame:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    frames = []
    for y in range(s.year, e.year + 1):
        for p in sorted((PANEL_DIR / f'year={y}').glob('*.parquet')):
            df = pd.read_parquet(p)
            df['date'] = pd.to_datetime(df['date'])
            df = df[(df['date'] >= s) & (df['date'] <= e)]
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _load_embed_years(start: str, end: str) -> pd.DataFrame:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    frames = []
    for y in range(s.year, e.year + 1):
        for p in sorted((EMBED_DIR / f'year={y}').glob('*.parquet')):
            df = pd.read_parquet(p, columns=['permno', 'date', 'vec'])
            df['date'] = pd.to_datetime(df['date'])
            df = df[(df['date'] >= s) & (df['date'] <= e)]
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _assemble(window_start: str, window_end: str, label: str):
    panel = _load_panel_years(window_start, window_end)
    embed = _load_embed_years(window_start, window_end)
    embed_pca = project_text_to_pca(embed, pca)
    X, y_excess, groups, meta = assemble_walk_features(panel, embed_pca)
    print(f'{label}: panel={len(panel):>7}  embed={len(embed):>7}  '
          f'X={X.shape}  groups={len(groups)} Fridays')
    return X, y_excess, groups, meta


X_train, y_train_excess, groups_train, meta_train = _assemble(TRAIN_START, TRAIN_END, 'train')
X_val,   y_val_excess,   groups_val,   meta_val   = _assemble(VAL_START,   VAL_END,   'val  ')
X_test,  y_test_excess,  groups_test,  meta_test  = _assemble(TEST_START,  TEST_END,  'test ')

# Drop zero-information columns based on TRAIN (carry the same drop to val/test).
# Some Sharadar fields (roa, roe, invcapavg, equityavg) are only in the ARY
# dimension; the panel uses ARQ so they're structurally NaN. Also drops dlret /
# dlstcd in windows with no delistings.
_before = X_train.shape[1]
X_train, X_val, X_test = drop_zero_info_columns(X_train, X_val, X_test)
print(f'dropped {_before - X_train.shape[1]} zero-info cols; '
      f'feature count now {X_train.shape[1]}')

# Lambdarank labels (32 buckets on per-date excess return).
buckets_train = compute_excess_return_buckets(meta_train, n_buckets=N_BUCKETS).astype(int).values
buckets_val   = compute_excess_return_buckets(meta_val,   n_buckets=N_BUCKETS).astype(int).values

print(f'first 8 feature cols: {list(X_train.columns[:8])}')
print(f'NaN-rich cols in train (top 5): '
      f'{X_train.isna().mean().sort_values(ascending=False).head(5).to_dict()}')

## C. Optuna HP search

15 TPE trials over `num_leaves`, `learning_rate`, `min_data_in_leaf`,
`feature_fraction`, `bagging_fraction`, `lambda_l2`. Objective: val NDCG@30.
MedianPruner kills slow-converging trials. Study persisted to
`artifacts/ranker/walk-001/optuna_study.pkl` for later inspection.

In [ ]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'num_leaves':       trial.suggest_int('num_leaves', 15, 127),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'lambda_l2':        trial.suggest_float('lambda_l2', 0.0, 5.0),
        'n_estimators':     500,
    }
    model = build_ranker(params)
    model.fit(
        X_train, buckets_train,
        group=groups_train,
        eval_set=[(X_val, buckets_val)],
        eval_group=[groups_val],
        eval_at=[TOP_K],
        callbacks=[early_stopping(stopping_rounds=30, verbose=False)],
    )
    return float(model.best_score_['valid_0'][f'ndcg@{TOP_K}'])


study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(),
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f'best NDCG@{TOP_K}: {study.best_value:.4f}')
print(f'best params: {json.dumps(study.best_params, indent=2)}')
joblib.dump(study, OUT_DIR / 'optuna_study.pkl')
print(f'study persisted -> {(OUT_DIR / "optuna_study.pkl").relative_to(repo_root())}')

## D. Final fit with best HPs + early stopping on val

In [ ]:
best_params = {**study.best_params, 'n_estimators': 2000}
model = build_ranker(best_params)
model.fit(
    X_train, buckets_train,
    group=groups_train,
    eval_set=[(X_val, buckets_val)],
    eval_group=[groups_val],
    eval_at=[TOP_K],
    callbacks=[early_stopping(stopping_rounds=50, verbose=True)],
)
best_iter = int(model.best_iteration_)
val_ndcg = float(model.best_score_['valid_0'][f'ndcg@{TOP_K}'])
print(f'best iteration: {best_iter}, val NDCG@{TOP_K}: {val_ndcg:.4f}')

## E. OOS evaluation + feature importance

Rank IC + decile spread + hit rate + top-K Jaccard on the held-out 2009 test
window. Sanity gate: `test rank IC mean > 0` (else abort — model isn't ranking).
Top-20 features by gain plotted inline.

In [ ]:
test_metrics = evaluate_ranker(model, X_test, y_test_excess, meta_test['date'],
                               top_k=TOP_K, entity_ids=meta_test['permno'])
print('test metrics:')
for k, v in test_metrics.items():
    print(f'  {k:>20}: {v:.4f}')

assert test_metrics['rank_ic_mean'] > 0, (
    f"sanity gate failed: test rank IC mean = {test_metrics['rank_ic_mean']:.4f} <= 0"
)

fi = pd.DataFrame({
    'feature': X_train.columns,
    'gain': model.booster_.feature_importance(importance_type='gain'),
}).sort_values('gain', ascending=False)
print('\ntop 20 features by gain:')
print(fi.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 7))
fi.head(20).iloc[::-1].plot.barh(x='feature', y='gain', ax=ax, legend=False)
ax.set_title(f'Top 20 features by gain (walk {WALK_ID})')
plt.tight_layout()
plt.show()

## E2. Alternative model: LGBMRegressor + fresh Optuna pass

Same X / y_excess / groups. Predict continuous excess return directly, then
rank the predictions. Fresh 15-trial Optuna search (same budget as cell C)
optimizing **val NDCG@30** — i.e. the same ranking objective the lambdarank
model targeted, so the comparison is apples-to-apples on what we actually care
about. Decision rule: keep regressor if its **test rank IC** beats lambdarank's.

**Verdict (walk 1):** lambdarank wins on both val NDCG@30 (0.233 vs 0.176) and
test rank IC (+0.0262 vs +0.0161). Regressor early-stopped at iter 5, meaning
val RMSE plateaued almost immediately — a structural sign that L2 regression
on raw cross-sectional excess return is dominated by tail observations.
Lambdarank's pairwise-within-group loss is naturally robust to those outliers
because once a stock is placed first, "more correct" doesn't move the loss.
**Conclusion: keep lambdarank in the walk-loop (cell G); regressor cell stays
in the notebook as a documented negative-result experiment.**

In [ ]:
from src.utils.ranker import build_regressor, compute_grouped_ndcg
from lightgbm import early_stopping as _es

# Continuous targets aligned to X.
y_train_arr = np.asarray(y_train_excess, dtype=np.float64)
y_val_arr   = np.asarray(y_val_excess,   dtype=np.float64)


def objective_reg(trial: optuna.Trial) -> float:
    params = {
        'num_leaves':       trial.suggest_int('num_leaves', 15, 127),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'lambda_l2':        trial.suggest_float('lambda_l2', 0.0, 5.0),
        'n_estimators':     500,
    }
    reg = build_regressor(params)
    reg.fit(
        X_train, y_train_arr,
        eval_set=[(X_val, y_val_arr)],
        callbacks=[_es(stopping_rounds=30, verbose=False)],
    )
    val_scores = reg.predict(X_val, num_iteration=reg.best_iteration_)
    return compute_grouped_ndcg(val_scores, buckets_val, groups_val, k=TOP_K)


study_reg = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(),
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study_reg.optimize(objective_reg, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
print(f'regressor best val NDCG@{TOP_K}: {study_reg.best_value:.4f}')
print(f'regressor best params: {json.dumps(study_reg.best_params, indent=2)}')

# Final fit with best HPs + early stopping.
best_params_reg = {**study_reg.best_params, 'n_estimators': 2000}
reg_model = build_regressor(best_params_reg)
reg_model.fit(
    X_train, y_train_arr,
    eval_set=[(X_val, y_val_arr)],
    callbacks=[_es(stopping_rounds=50, verbose=False)],
)
reg_best_iter = int(reg_model.best_iteration_)
reg_val_scores = reg_model.predict(X_val, num_iteration=reg_best_iter)
reg_val_ndcg = compute_grouped_ndcg(reg_val_scores, buckets_val, groups_val, k=TOP_K)
print(f'regressor: best_iter={reg_best_iter}, val NDCG@{TOP_K}={reg_val_ndcg:.4f}')

# OOS test eval (reuse evaluate_ranker — it just calls .predict()).
reg_test_metrics = evaluate_ranker(reg_model, X_test, y_test_excess, meta_test['date'],
                                   top_k=TOP_K, entity_ids=meta_test['permno'])
print('\nregressor test metrics:')
for k, v in reg_test_metrics.items():
    print(f'  {k:>20}: {v:.4f}')

# Side-by-side comparison.
comp = pd.DataFrame({
    'lambdarank': {**test_metrics, 'val_ndcg_30': val_ndcg, 'best_iter': best_iter},
    'regressor':  {**reg_test_metrics, 'val_ndcg_30': reg_val_ndcg, 'best_iter': reg_best_iter},
})
print('\n=== lambdarank vs regressor (walk 1) ===')
print(comp.to_string())
winner = 'regressor' if reg_test_metrics['rank_ic_mean'] > test_metrics['rank_ic_mean'] else 'lambdarank'
delta = reg_test_metrics['rank_ic_mean'] - test_metrics['rank_ic_mean']
print(f"\nwinner by test rank IC: {winner} (delta = {delta:+.4f})")

## F. Persist artifacts

Writes `model.joblib` (model + feature names), `hp.json`, `summary.json`, and
`feature_importance.csv`. The `optuna_study.pkl` was already persisted in cell C.
Walk-1 is now complete; downstream notebook 07 (RL) can consume `model.joblib`.

In [ ]:
joblib.dump(
    {'model': model, 'feature_names': X_train.columns.tolist()},
    OUT_DIR / 'model.joblib',
)
(OUT_DIR / 'hp.json').write_text(json.dumps({
    **study.best_params,
    f'val_ndcg_at_{TOP_K}': val_ndcg,
    'best_iteration': best_iter,
}, indent=2))
fi.to_csv(OUT_DIR / 'feature_importance.csv', index=False)

summary = {
    'walk_id': WALK_ID,
    'train_window': [TRAIN_START, TRAIN_END],
    'val_window':   [VAL_START,   VAL_END],
    'test_window':  [TEST_START,  TEST_END],
    'n_features': int(X_train.shape[1]),
    'n_pca': int(n_pca),
    'n_train_rows': int(len(X_train)),
    'n_val_rows':   int(len(X_val)),
    'n_test_rows':  int(len(X_test)),
    'best_iteration': best_iter,
    f'val_ndcg_at_{TOP_K}': val_ndcg,
    **{f'test_{k}': float(v) for k, v in test_metrics.items()},
    'passed_sanity': bool(test_metrics['rank_ic_mean'] > 0),
}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))

print(f'wrote artifacts to {OUT_DIR.relative_to(repo_root())}/')
for p in sorted(OUT_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')
print('\nsummary:')
print(json.dumps(summary, indent=2))

## G. Walk-forward loop (walks 2-16)

Per spec §7.2, HPs are frozen after walk 1 and reused for all subsequent walks.
For each walk N: train 2002 → (2007+N-1), val (2008+N-1), test (2009+N-1).
Resumable — existing `artifacts/ranker/walk-{N:03d}/summary.json` skips the walk.

In [ ]:
ARTIFACTS_ROOT = repo_root() / 'artifacts'
WALK_RANGE = range(2, 17)  # walks 2..16; walk 1 done in cells A-F
FROZEN_HPS = {**study.best_params, 'n_estimators': 2000}  # from walk 1's Optuna
print(f'frozen HPs (from walk 1): {json.dumps(study.best_params, indent=2)}')


def _walk_windows(walk_id: int) -> tuple[str, str, str, str, str, str]:
    """Walk N: train 2002-01-01..(2007+N-1)-12-31, val (2008+N-1), test (2009+N-1)."""
    train_end_year = 2007 + walk_id - 1
    return (
        '2002-01-01', f'{train_end_year}-12-31',
        f'{train_end_year + 1}-01-01', f'{train_end_year + 1}-12-31',
        f'{train_end_year + 2}-01-01', f'{train_end_year + 2}-12-31',
    )


def process_walk(walk_id: int) -> dict:
    out_dir = ARTIFACTS_ROOT / 'ranker' / f'walk-{walk_id:03d}'
    summary_path = out_dir / 'summary.json'
    if summary_path.exists():
        existing = json.loads(summary_path.read_text())
        print(f'walk {walk_id:>2}: exists, test_rank_ic_mean='
              f'{existing.get("test_rank_ic_mean", float("nan")):.4f} — skipping')
        return existing

    out_dir.mkdir(parents=True, exist_ok=True)
    tr_s, tr_e, vl_s, vl_e, te_s, te_e = _walk_windows(walk_id)
    walk_pca, walk_n_pca = load_walk_pca(walk_id)

    def _build(ws: str, we: str):
        panel = _load_panel_years(ws, we)
        embed = _load_embed_years(ws, we)
        embed_pca = project_text_to_pca(embed, walk_pca)
        return assemble_walk_features(panel, embed_pca)

    Xtr, ytr, gtr, mtr = _build(tr_s, tr_e)
    Xvl, yvl, gvl, mvl = _build(vl_s, vl_e)
    Xte, yte, gte, mte = _build(te_s, te_e)

    # Drop zero-info columns based on this walk's train.
    Xtr, Xvl, Xte = drop_zero_info_columns(Xtr, Xvl, Xte)

    btr = compute_excess_return_buckets(mtr, n_buckets=N_BUCKETS).astype(int).values
    bvl = compute_excess_return_buckets(mvl, n_buckets=N_BUCKETS).astype(int).values

    walk_model = build_ranker(FROZEN_HPS)
    walk_model.fit(
        Xtr, btr, group=gtr,
        eval_set=[(Xvl, bvl)], eval_group=[gvl], eval_at=[TOP_K],
        callbacks=[early_stopping(stopping_rounds=50, verbose=False)],
    )
    walk_best_iter = int(walk_model.best_iteration_)
    walk_val_ndcg = float(walk_model.best_score_['valid_0'][f'ndcg@{TOP_K}'])
    walk_test_metrics = evaluate_ranker(walk_model, Xte, yte, mte['date'],
                                        top_k=TOP_K, entity_ids=mte['permno'])

    joblib.dump(
        {'model': walk_model, 'feature_names': Xtr.columns.tolist()},
        out_dir / 'model.joblib',
    )
    walk_fi = pd.DataFrame({
        'feature': Xtr.columns,
        'gain': walk_model.booster_.feature_importance(importance_type='gain'),
    }).sort_values('gain', ascending=False)
    walk_fi.to_csv(out_dir / 'feature_importance.csv', index=False)

    walk_summary = {
        'walk_id': walk_id,
        'train_window': [tr_s, tr_e],
        'val_window':   [vl_s, vl_e],
        'test_window':  [te_s, te_e],
        'n_features': int(Xtr.shape[1]),
        'n_pca': int(walk_n_pca),
        'n_train_rows': int(len(Xtr)),
        'n_val_rows':   int(len(Xvl)),
        'n_test_rows':  int(len(Xte)),
        'best_iteration': walk_best_iter,
        f'val_ndcg_at_{TOP_K}': walk_val_ndcg,
        **{f'test_{k}': float(v) for k, v in walk_test_metrics.items()},
        'passed_sanity': bool(walk_test_metrics['rank_ic_mean'] > 0),
        'hps_frozen_from_walk_1': True,
    }
    summary_path.write_text(json.dumps(walk_summary, indent=2))
    print(f'walk {walk_id:>2}: train={len(Xtr):>6}  val={len(Xvl):>5}  test={len(Xte):>5}  '
          f'val_ndcg={walk_val_ndcg:.4f}  test_rank_ic={walk_test_metrics["rank_ic_mean"]:+.4f}  '
          f'sanity={"PASS" if walk_summary["passed_sanity"] else "FAIL"}')
    return walk_summary


walk_summaries = [process_walk(w) for w in WALK_RANGE]
print(f'\nfinished walks {min(WALK_RANGE)}..{max(WALK_RANGE)} '
      f'({sum(1 for s in walk_summaries if s.get("passed_sanity")) }/{len(walk_summaries)} passed sanity)')

## H. Cross-walk diagnostics

Aggregate all 16 walks' `summary.json` into one table; plot test rank IC over
walks to spot regime breaks or model drift. Saves the consolidated summary to
`artifacts/ranker/all_walks_summary.json` for notebook 07 (RL) to read once.

In [ ]:
all_walks = []
for w in range(1, 17):
    p = ARTIFACTS_ROOT / 'ranker' / f'walk-{w:03d}' / 'summary.json'
    if p.exists():
        all_walks.append(json.loads(p.read_text()))

walks_df = pd.DataFrame(all_walks).sort_values('walk_id').reset_index(drop=True)
display_cols = [
    'walk_id', 'n_train_rows', 'n_test_rows', 'best_iteration',
    f'val_ndcg_at_{TOP_K}', 'test_rank_ic_mean', 'test_rank_ic_ir',
    'test_decile_spread_bps', 'test_hit_rate', 'test_top_k_jaccard',
    'passed_sanity',
]
print('per-walk summary:')
print(walks_df[display_cols].to_string(index=False))

(ARTIFACTS_ROOT / 'ranker' / 'all_walks_summary.json').write_text(
    json.dumps({'walks': all_walks}, indent=2)
)

# Plot test rank IC over walks (with test-window labels on x-axis).
fig, ax = plt.subplots(figsize=(10, 4))
test_years = [s['test_window'][0][:4] for s in all_walks]
ax.bar(test_years, walks_df['test_rank_ic_mean'], color='steelblue')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('test year')
ax.set_ylabel('test rank IC mean')
ax.set_title('Walk-forward test rank IC (per walk)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()